# 16 — Evaluation-Driven Prompt Optimization

    ## Scenario and success criteria

    Northstar fixes ticker normalization without tuning on the final test set.

    This guided lab succeeds when its assertions pass and the learner can explain why the baseline fails, what the mitigation changes, and which production controls remain outside the simulation.

    ## Learning objectives

    - Separate development and holdout data.
- Score component and joint outcomes.
- Reject a local fix that causes a global regression.

    **Prerequisites:** Courses 01–13 and the preceding advanced/enterprise lesson.
    **Safety boundary:** all behavior is deterministic and synthetic; there are no credentials, external calls, or side effects. Printed results are simulation evidence, not a live-model benchmark.

## Mental model and architecture

![Course 16 architecture](diagram-1.svg)

Treat the model as one uncertain component inside a deterministic control plane. Inputs, schemas, identity, authorization, metrics, release gates, and state transitions remain application responsibilities.

## Baseline and failure injection

Optimizing on one visible failure leaks evaluation data and can overfit the prompt.

The next cell defines the synthetic fixture and the smallest reusable primitive needed to make that failure observable.

In [ ]:
from lab16 import ExtractionCase, alias_policy, baseline_policy, score, split_cases

cases = [
    ExtractionCase("d1", "$GOOG rallies on AI", "Google", "positive", "development"),
    ExtractionCase("d2", "Northstar posts profit", "Northstar", "positive", "development"),
    ExtractionCase("h1", "$MSFT reports growth", "Microsoft", "positive", "holdout"),
    ExtractionCase("h2", "Acme faces lawsuit", "Acme", "negative", "holdout"),
]

## Experiment

Run the baseline and candidate on the same fixture so the comparison is attributable.

In [ ]:
for split in ("development", "holdout"):
    rows = split_cases(cases, split)
    print(split, score("baseline", baseline_policy, rows))
    print(split, score("alias", alias_policy, rows))

## Evaluation

The assertions below are the executable contract. They validate both a positive path and a boundary or failure path; a printed claim alone is not proof.

In [ ]:
development = score("alias", alias_policy, split_cases(cases, "development"))
holdout = score("alias", alias_policy, split_cases(cases, "holdout"))
assert development.joint_accuracy == 1.0
assert holdout.joint_accuracy == 1.0
assert {case.case_id for case in split_cases(cases, "development")}.isdisjoint({case.case_id for case in split_cases(cases, "holdout")})

## Production upgrade

Freeze the contract and holdout before searching. Version the prompt, dataset, model settings, and metric code together; release only after slice and safety gates pass.

| Teaching lab | Production system |
| --- | --- |
| Synthetic fixtures | Versioned, reviewed, privacy-safe datasets |
| Deterministic simulation | Provider adapter plus optional recorded replay |
| In-process state | Durable state with tenant and retention boundaries |
| Assertions | CI gates, staged rollout, monitoring, and rollback |

## Exercises

1. Add one normal, one boundary, and one adversarial case without weakening an invariant.
2. Change one design variable and report the metric numerator, denominator, unit, and direction.
3. Write a production decision memo that identifies owner, failure policy, monitoring signal, and rollback trigger.

## Takeaway

Use probabilistic components for bounded interpretation; use trusted deterministic code for permissions, validation, metrics, and consequential state changes.